In [1]:
!pip install -q ultralytics roboflow opencv-python-headless gradio

## **Config**

In [2]:
from roboflow import Roboflow

rf = Roboflow(api_key="b5LYnIRWuUywAq4uui5s")
project = rf.workspace("rzeszow-university-of-technology").project("solar-panels-seg")
dataset = project.version(2).download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Solar-panels-seg-2 in yolov8:: 100%|██████████| 19240/19240 [00:03<00:00, 5180.40it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [ ]:
from ultralytics import YOLO

# Load pre-trained YOLO11 segmentation model
model = YOLO("yolo11s-seg.pt")

# Train on your dataset
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=40,
    imgsz=640,
    batch=15,
    project="solar_project",
    name="yolo11_seg"
)


Ultralytics 8.4.214 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=15, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Solar-panels-seg-2/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11_seg, nbs=64, nms=None, opset=None, optim

In [ ]:
import gradio as gr
import cv2
from ultralytics import YOLO


# Load your newly trained best weights
MODEL_PATH = "/content/best.pt"
trained_model = YOLO(MODEL_PATH)

def detect_solar_panels(image, conf_threshold, meters_per_pixel):
    if image is None:
        return None, "Please upload an image."

    # Run inference on the uploaded RGB image
    results = trained_model.predict(source=image, conf=conf_threshold, verbose=False)
    r = results[0]

    # Count total detected solar arrays/panels
    total_count = len(r.boxes)

    # Calculate total polygon pixel area if masks are present
    total_pixels = 0
    if r.masks is not None:
        for mask in r.masks.data:
            total_pixels += mask.sum().item()

    # Estimate real-world area (m²) and kW capacity (~0.20 kW per m² for standard panels)
    area_sqm = total_pixels * (meters_per_pixel ** 2)
    estimated_kw = area_sqm * 0.20

    # Generate annotated image with polygon masks & bounding boxes
    annotated_bgr = r.plot()
    annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

    # Format summary report
    summary = (
        f"### Detection Summary\n"
        f"- **Total Panels / Arrays Detected:** {total_count}\n"
        f"- **Total Masked Area:** {int(total_pixels):,} pixels (~{area_sqm:.2f} m²)\n"
        f"- **Estimated System Capacity:** ~{estimated_kw:.2f} kW\n"
    )

    if total_count == 0:
        summary += "\n*Tip: Try lowering the Confidence Threshold slider if panels are faint.*"

    return annotated_rgb, summary

# Build Gradio interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ☀️ Rooftop Solar Panel Detection & Counting")
    gr.Markdown("Upload a satellite or drone rooftop image to detect solar arrays, count units, and estimate power capacity.")

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(type="numpy", label="Upload Rooftop Image")
            conf_slider = gr.Slider(
                minimum=0.10, maximum=0.90, value=0.40, step=0.05,
                label="Confidence Threshold"
            )
            scale_slider = gr.Slider(
                minimum=0.02, maximum=0.50, value=0.15, step=0.01,
                label="Image Resolution (Meters per Pixel — ~0.15 for Satellite, ~0.03 for Drone)"
            )
            run_btn = gr.Button("Detect Solar Panels", variant="primary")

        with gr.Column():
            output_image = gr.Image(type="numpy", label="Detected Solar Panels")
            output_text = gr.Markdown(label="Results")

    run_btn.click(
        fn=detect_solar_panels,
        inputs=[input_image, conf_slider, scale_slider],
        outputs=[output_image, output_text]
    )

# Launch with share=True to generate a public Colab link
demo.launch(share=True, debug=True)


The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4a5613c68b74dafd98.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
